# [부록 B] 2단원 전체 실습 코드 (한 번에 실행용)
단계별 설명은 `RPS_finetuning_step_by_step.ipynb`와 교안 2.2를 보세요.

In [ ]:
# ===== 0. 환경 확인 =====
!nvidia-smi

In [ ]:
# ===== 1. 데이터 다운로드 (Colab 보안 비밀에 ROBOFLOW_API_KEY 등록 필요) =====
!pip install -q roboflow
from roboflow import Roboflow
from google.colab import userdata
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
rf.workspace('roboflow-58fyf').project('rock-paper-scissors-sxsw')   .version(14).download('yolov11', location='/content/RockPaperScissors_Data')
import zipfile          # STEP 8(결과 압축)에서 사용

In [ ]:
# ===== 2. yaml 생성 =====
import yaml
data = {'train': '/content/RockPaperScissors_Data/train/images/',
        'val'  : '/content/RockPaperScissors_Data/valid/images/',
        'test' : '/content/RockPaperScissors_Data/test/images',
        'names': ['Paper', 'Rock', 'Scissors'],
        'nc'   : 3}
YAML = '/content/RockPaperScissors_Data/RockPaperScissors_Data.yaml'
with open(YAML, 'w') as f:
    yaml.dump(data, f)

In [ ]:
# ===== 3. 설치 & 모델 로드 =====
!pip install ultralytics
from ultralytics import YOLO
from ultralytics.utils.plotting import colors
colors.palette[2] = (252, 109, 47)   # 2번 클래스(Scissors) 기본색이 거의 흰색 → 주황색 (RGB)
model = YOLO('yolo11n.pt')

In [ ]:
# ===== 4. 학습 =====
model.train(data=YAML, epochs=30, patience=5, imgsz=416)
print(model.names)          # {0: 'Paper', 1: 'Rock', 2: 'Scissors'}

In [ ]:
# ===== 5. 추론 =====
results = model(source='/content/RockPaperScissors_Data/test/images/', save=True)
save_dir = str(results[0].save_dir)
print('결과 저장 위치:', save_dir)

In [ ]:
# ===== 6. 압축 & 다운로드 =====
import glob, os
os.makedirs('/content/detected_result/', exist_ok=True)
with zipfile.ZipFile('/content/detected_result/detected_images.zip', 'w') as zf:
    for path in glob.glob(save_dir + '/*'):
        zf.write(path)

from google.colab import files
files.download('/content/detected_result/detected_images.zip')
files.download('/content/runs/detect/train/weights/best.pt')